## 0. Setup

In [12]:
### Main Modules
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Models
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

### Metrics and calibrarion
from sklearn.metrics import (f1_score,accuracy_score,recall_score,precision_score,roc_auc_score,roc_curve,precision_recall_curve,confusion_matrix)

### Import conformal prediction wrapper
from crepes import WrapClassifier

## 1. Importar objetos

In [13]:
### Cargar modelo original
modelo_original = XGBClassifier()
modelo_original.load_model('../../01_data/output/01_original_r_trained_model_xgboost.ubj')

### Cargar modelo reducido (10 variables)
modelo_top_10 = XGBClassifier()
modelo_top_10.load_model('../../04_sub_model/output/01_top_10.ubj')

### Cargar datos
train = pd.read_csv('../../01_data/output/01_original_data_train.csv')
test = pd.read_csv('../../01_data/output/01_original_data_test.csv')
validation = pd.read_csv('../../01_data/output/01_original_data_validation.csv')

### Limpiar dependiente
for df in [train,test,validation]:
    df["pobre"] = np.where(df["pobre"] == "Yes", 1, 0) 

### Split datos
train_x, train_y = train.drop(columns=["id","pobre"]), train["pobre"]
validation_x, validation_y = validation.drop(columns=["id","pobre"]), validation["pobre"]
test_x, test_y = test.drop(columns=["id","pobre"]), test["pobre"]

### Crear lista con 10 variables mas importantes del modelo grande (SHAP)
top_10_predictors = [
    "oc_household_average", "arriendo", "regimen_salud_1_household_average",
    "tipo_propiedad_vivienda_hogar", "ocupado_horas_trabajadas_normalmente_household_working",
    "numero_personas_unidad_gasto", "tipo_propiedad_vivienda_hogar_3",
    "ocupado_tamano_de_la_empresa_9_household_working", 
    "maximo_nivel_educativo_6_household_average", "ocupado_relab_4_household_working"
]

## 2. Crear clasificador conformal

In [14]:
## Setup modelo estándar (152 variables)
model = WrapClassifier(XGBClassifier(
                        n_estimators=500,
                        max_depth=7,
                        learning_rate=0.04,
                        min_child_weight=25,
                        gamma=0,
                        colsample_bytree=0.6,
                        subsample=0.8,
                        random_state=123,
                        eval_metric='logloss'))

## Setup modelo restringido (10 Variables)
model_top_10 = WrapClassifier(XGBClassifier(
                              n_estimators=500,
                              max_depth=7,
                              learning_rate=0.04,
                              min_child_weight=25,
                              gamma=0,
                              colsample_bytree=0.6,
                              subsample=0.8,
                              random_state=123,
                              eval_metric='logloss'))

### Ajustar modelos con los datos de entrenamiento
model.fit(train_x, train_y)
model_top_10.fit(train_x[top_10_predictors],train_y)

### Calibrar modelo con los datos de validación
model.calibrate(validation_x, validation_y)
model_top_10.calibrate(validation_x[top_10_predictors], validation_y)

### Evaluar modelos para un alfa de 0.05
print(model.evaluate(test_x, test_y, confidence=0.95))
print(model_top_10.evaluate(test_x[top_10_predictors], test_y, confidence=0.95))


{'error': 0.0505845396547816, 'avg_c': 1.1538586094109264, 'one_c': 0.8461413905890734, 'empty': 0.0, 'ks_test': 0.09238195058792731, 'time_fit': 0.0, 'time_evaluate': 30.7806077003479}
{'error': 0.050940768807280046, 'avg_c': 1.201852391592992, 'one_c': 0.798147608407008, 'empty': 0.0, 'ks_test': 0.17039060484196922, 'time_fit': 0.0, 'time_evaluate': 27.6421160697937}


### 3. Hacer predicciones

In [15]:
### Predicciones del modelo base
predictions = test[["pobre"]]
predictions_top_10_model = predictions.assign(prediction_top_10_model = (model_top_10.predict_proba(test_x[top_10_predictors])[:,1] >= 0.34).astype(int))
prediction_main_model = predictions.assign(prediction_main_model = (model.predict_proba(test_x)[:,1] >= 0.34).astype(int))

### Realizar predicciones con el modelo calibrado
pred_sets_main_model = model.predict_set(test_x, confidence=0.95)
pred_sets_top_10_model = model_top_10.predict_set(test_x[top_10_predictors], confidence=0.95)

In [16]:
### Marcar los conjuntos conformales
def define_conformal_set(df,conformal_set):

    ### Create boolean flags for each class
    df['contains_class_0'] = [0 in p_set for p_set in conformal_set]
    df['contains_class_1'] = [1 in p_set for p_set in conformal_set]

    ### Computing set size 
    df['set_size'] = [len(p_set) for p_set in conformal_set]

    ### Create indicator of prediction set

    # Conditions list
    condition_list = [
        (df["contains_class_0"] == True) & (df["contains_class_1"] == False),
        (df["contains_class_0"] == False) & (df["contains_class_1"] == True),
        (df["set_size"] == 2)
    ]

    # Choice list
    choice_list = ["0", "1", "(0,1)"]

    # Pass through conditional
    df["conformal_set"] = np.select(condition_list, choice_list, default="empty")

    return df

### Apply function
conformal_set_main_model = define_conformal_set(df=prediction_main_model,conformal_set=pred_sets_main_model).assign(model='Principal (p = 152)')
conformal_set_top_10_model = define_conformal_set(df=predictions_top_10_model,conformal_set=pred_sets_top_10_model).assign(model='Restringido (p = 10)')

### Bind together
predictions = pd.concat([conformal_set_main_model,conformal_set_top_10_model])


## 4. Construir tabla de predicciones

In [21]:
### Make table
tab_count = predictions.groupby(["pobre","model"]).conformal_set.value_counts().reset_index()
tab_share = predictions.groupby(["pobre","model"]).conformal_set.value_counts(normalize=True).reset_index()
tab = pd.merge(left=tab_count,right=tab_share,how='inner',on=["model","pobre","conformal_set"])

### Reorder table
tab.sort_values(["model",'pobre','conformal_set'],ascending=True,inplace=True)

### Change scale of variable
tab["proportion"] = tab["proportion"] * 100

### Change 
tab["pobre"] = tab["pobre"].case_when([(tab["pobre"] == 0,'No Pobre'),
                                       (tab["pobre"] == 1,'Pobre')])

### Rename table
tab = (tab.
       rename(columns = {'pobre':"Etiqueta real",
                         'model':'Modelo',
                         "conformal_set":"Conjunto Conformal",
                         "count":"Conteo",
                         "proportion":"Proporción"}).
       reindex(columns=["Modelo","Etiqueta real","Conjunto Conformal","Conteo","Proporción"]).
       round(2))

### Set the columns as a MultiIndex
tab = tab.set_index(['Modelo', 'Etiqueta real'])

### 2. Sort the index
tab = tab.sort_index()



## 5. Guardar tabla

In [ ]:
tab.to_latex('../../06_uncertainy/output/02_conformal_prediction.tex',float_format="{:.2f}".format,escape=True,index=True,multirow=True)